# Geohash-based Environment Classification Pipeline

Optimized for 40M+ POIs with:
- Vectorized geohash encoding
- Parallel processing
- Chunked memory-efficient processing
- NumPy optimizations

In [ ]:
import pandas as pd
import numpy as np
import pygeohash as pgh
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import joblib
from joblib import Parallel, delayed
from tqdm import tqdm
import warnings
import gc
warnings.filterwarnings('ignore')

# For even faster geohash encoding
try:
    import geohash2 as gh2
    USE_GEOHASH2 = True
except ImportError:
    USE_GEOHASH2 = False
    print("Install geohash2 for faster encoding: pip install geohash2")

In [ ]:
class FastUnsupervisedEnvironmentClassifier:
    """
    Optimized Unsupervised Environment Classification for 40M+ POIs
    Improved to handle highway/transportation dominance
    """
    
    def __init__(self, geohash_precision=6, n_clusters=7, n_jobs=-1, chunk_size=500000):
        self.precision = geohash_precision
        self.n_clusters = n_clusters
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size
        self.poi_categories = [
            'transportation', 'public_services', 'business', 
            'tourism', 'infrastructure', 'residential', 'other'
        ]
        self.category_to_idx = {cat: i for i, cat in enumerate(self.poi_categories)}
        
        # Category weights to reduce transportation dominance
        # Lower weight = less influence on clustering
        self.category_weights = {
            'transportation': 0.3,  # Downweight highways
            'public_services': 1.0,
            'business': 1.2,        # Slightly boost business
            'tourism': 1.2,
            'infrastructure': 0.5,  # Downweight infrastructure
            'residential': 1.0,
            'other': 0.5
        }
        
        self.scaler = StandardScaler()
        self.clustering_model = None
        self.cell_features = None
        self.cell_clusters = {}
        self.cluster_names = {}
        
    def _encode_geohash_batch(self, lats, lons):
        """Vectorized geohash encoding"""
        if USE_GEOHASH2:
            return [gh2.encode(lat, lon, self.precision) for lat, lon in zip(lats, lons)]
        else:
            return [pgh.encode(lat, lon, precision=self.precision) for lat, lon in zip(lats, lons)]
    
    def _encode_geohash_parallel(self, df):
        """Parallel geohash encoding for large datasets"""
        n_chunks = max(1, len(df) // self.chunk_size)
        chunks = np.array_split(df.index, n_chunks)
        
        def process_chunk(indices):
            chunk_df = df.loc[indices]
            return self._encode_geohash_batch(
                chunk_df['latitude'].values,
                chunk_df['longitude'].values
            )
        
        results = Parallel(n_jobs=self.n_jobs, backend='threading')(
            delayed(process_chunk)(chunk) for chunk in tqdm(chunks, desc="Encoding geohashes")
        )
        
        return [gh for result in results for gh in result]
    
    def _get_neighbor_geohashes(self, geohash):
        """Get all 8 neighboring geohash cells"""
        neighbors = []
        directions = ['top', 'bottom', 'right', 'left', 
                     'topleft', 'topright', 'bottomleft', 'bottomright']
        
        for direction in directions:
            try:
                neighbor = pgh.get_adjacent(geohash, direction)
                neighbors.append(neighbor)
            except:
                pass
        
        return neighbors
    
    def aggregate_poi_by_geohash(self, poi_df):
        """Optimized POI aggregation with category weighting"""
        print("Aggregating POIs by geohash (optimized)...")
        
        poi_df = poi_df.copy()
        geohash_col = f'geohash_{self.precision}'
        
        # Fast parallel geohash encoding
        print(f"Encoding {len(poi_df):,} POIs...")
        poi_df[geohash_col] = self._encode_geohash_parallel(poi_df)
        
        # Convert category to numeric for faster aggregation
        poi_df['category_idx'] = poi_df['category'].map(self.category_to_idx).fillna(6).astype(int)
        
        # Pre-aggregate counts using groupby (much faster than loop)
        print("Pre-aggregating POI counts...")
        
        # Get POI counts per cell and category
        poi_counts = poi_df.groupby([geohash_col, 'category_idx']).size().unstack(fill_value=0)
        
        # Ensure all categories exist
        for i in range(len(self.poi_categories)):
            if i not in poi_counts.columns:
                poi_counts[i] = 0
        poi_counts = poi_counts[sorted(poi_counts.columns)]
        
        # Apply category weights
        weights = np.array([self.category_weights[cat] for cat in self.poi_categories])
        poi_counts_weighted = poi_counts * weights
        
        # Get total POIs per cell (weighted and unweighted)
        total_pois = poi_counts.sum(axis=1)
        total_pois_weighted = poi_counts_weighted.sum(axis=1)
        
        # Create neighbor lookup
        print("Building neighbor lookup...")
        unique_cells = poi_counts.index.tolist()
        cell_set = set(unique_cells)
        
        # Build neighbor POI counts efficiently
        neighbor_counts = {}
        for cell in tqdm(unique_cells, desc="Computing neighbor features"):
            neighbors = self._get_neighbor_geohashes(cell)
            neighbor_sum = np.zeros(len(self.poi_categories))
            for n in neighbors:
                if n in cell_set:
                    neighbor_sum += poi_counts_weighted.loc[n].values
            neighbor_counts[cell] = neighbor_sum
        
        # Extract features for all cells (vectorized)
        print("Extracting features...")
        cell_data = []
        
        for cell in tqdm(unique_cells, desc="Building feature vectors"):
            counts = poi_counts.loc[cell].values.astype(float)
            counts_weighted = poi_counts_weighted.loc[cell].values.astype(float)
            total = total_pois[cell]
            total_weighted = total_pois_weighted[cell]
            neighbor_sum = neighbor_counts[cell]
            neighbor_total = neighbor_sum.sum()
            
            center = pgh.decode(cell)
            features = self._extract_cell_features_fast(
                counts, counts_weighted, total, total_weighted, 
                neighbor_sum, neighbor_total, weights
            )
            
            cell_data.append({
                'geohash': cell,
                'latitude': center[0],
                'longitude': center[1],
                'features': features,
                'total_pois': total,
                'raw_counts': counts  # Store for analysis
            })
        
        print(f"Created features for {len(cell_data):,} cells")
        return pd.DataFrame(cell_data)
    
    def _extract_cell_features_fast(self, poi_counts, poi_counts_weighted, total_pois, 
                                     total_weighted, neighbor_counts, neighbor_total, weights):
        """Optimized feature extraction with weighting to handle transportation dominance"""
        features = []
        
        # 1. Weighted POI Type Distribution (7 features)
        poi_ratios_weighted = poi_counts_weighted / (total_weighted + 1e-10)
        features.extend(poi_ratios_weighted)
        
        # 2. Total POI Density (excluding transportation-heavy influence)
        non_transport_total = total_pois - poi_counts[0] * 0.7  # Reduce transport impact
        features.append(np.log1p(max(0, non_transport_total)))
        
        # 3. POI Diversity Metrics (3 features)
        if total_pois > 0:
            # Shannon entropy on weighted counts
            p = poi_counts_weighted / (total_weighted + 1e-10)
            p = p[p > 0]
            entropy = -np.sum(p * np.log(p)) if len(p) > 0 else 0
            features.append(entropy)
            
            # Dominant POI ratio (weighted)
            features.append(np.max(poi_counts_weighted) / (total_weighted + 1e-10))
            
            # Number of POI types present
            features.append(np.sum(poi_counts > 0) / len(self.poi_categories))
        else:
            features.extend([0, 0, 0])
        
        # 4. Non-transportation activity score (key feature!)
        non_transport_pois = poi_counts[1:].sum()  # Exclude transportation
        non_transport_ratio = non_transport_pois / (total_pois + 1e-10)
        features.append(non_transport_ratio)
        
        # 5. POI Type Ratios (6 features) - More nuanced
        if total_pois > 0:
            # Business to residential ratio
            features.append(np.log1p((poi_counts[2] + 1) / (poi_counts[5] + 1)))
            
            # Tourism intensity (boosted)
            features.append(poi_counts[3] * 1.5 / (total_pois + 1e-10))
            
            # Public services density
            features.append(poi_counts[1] / (total_pois + 1e-10))
            
            # Business density (key indicator)
            features.append(poi_counts[2] / (total_pois + 1e-10))
            
            # Residential density
            features.append(poi_counts[5] / (total_pois + 1e-10))
            
            # Activity diversity (non-transport categories)
            active_categories = np.sum(poi_counts[1:] > 0)
            features.append(active_categories / 6)
        else:
            features.extend([0, 0, 0, 0, 0, 0])
        
        # 6. Neighborhood Context (7 features) - weighted
        neighbor_ratios = neighbor_counts / (neighbor_total + 1e-10)
        features.extend(neighbor_ratios)
        
        # 7. Spatial Contrast (7 features)
        contrast = poi_ratios_weighted - neighbor_ratios
        features.extend(contrast)
        
        # 8. Urban activity indicators (3 features)
        # Commercial activity score
        commercial_score = (poi_counts[2] + poi_counts[3] + poi_counts[1]) / (total_pois + 1e-10)
        features.append(commercial_score)
        
        # Residential character score
        residential_score = poi_counts[5] / (total_pois + 1e-10)
        features.append(residential_score)
        
        # Infrastructure ratio (helps identify industrial areas)
        infra_score = poi_counts[4] / (total_pois + 1e-10)
        features.append(infra_score)
        
        return np.array(features)
    
    def find_optimal_clusters(self, cell_df, max_clusters=15):
        """Find optimal clusters using MiniBatchKMeans for speed"""
        print("\nFinding optimal number of clusters...")
        
        X = np.vstack(cell_df['features'].values)
        X_scaled = self.scaler.fit_transform(X)
        
        # Use subset for optimization if dataset is large
        if len(X_scaled) > 10000:
            sample_idx = np.random.choice(len(X_scaled), 10000, replace=False)
            X_sample = X_scaled[sample_idx]
        else:
            X_sample = X_scaled
        
        silhouette_scores = []
        K_range = range(3, max_clusters + 1)
        
        for k in tqdm(K_range, desc="Testing clusters"):
            kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=1024, n_init=3)
            labels = kmeans.fit_predict(X_sample)
            silhouette_scores.append(silhouette_score(X_sample, labels))
        
        optimal_k = K_range[np.argmax(silhouette_scores)]
        print(f"Optimal clusters: {optimal_k}")
        
        return optimal_k
    
    def cluster_cells(self, cell_df, method='minibatch_kmeans', n_clusters=None):
        """Cluster cells using MiniBatchKMeans for speed"""
        if n_clusters is None:
            n_clusters = self.n_clusters
        
        print(f"\nClustering {len(cell_df):,} cells using {method}...")
        
        X = np.vstack(cell_df['features'].values)
        X_scaled = self.scaler.fit_transform(X)
        
        if method == 'minibatch_kmeans':
            self.clustering_model = MiniBatchKMeans(
                n_clusters=n_clusters,
                random_state=42,
                batch_size=1024,
                n_init=10
            )
        else:
            self.clustering_model = KMeans(
                n_clusters=n_clusters,
                random_state=42,
                n_init=10
            )
        
        labels = self.clustering_model.fit_predict(X_scaled)
        cell_df['cluster'] = labels
        
        # Create lookup dict
        self.cell_clusters = dict(zip(cell_df['geohash'], cell_df['cluster']))
        
        self._analyze_clusters(cell_df)
        print(f"Clustering completed! Found {len(np.unique(labels))} clusters")
        
        return cell_df
    
    def _analyze_clusters(self, cell_df):
        """Analyze and name clusters based on weighted characteristics"""
        print("\nAnalyzing cluster characteristics...")
        
        for cluster_id in sorted(cell_df['cluster'].unique()):
            if cluster_id == -1:
                continue
            
            cluster_cells = cell_df[cell_df['cluster'] == cluster_id]
            features_matrix = np.vstack(cluster_cells['features'].values)
            avg_features = np.mean(features_matrix, axis=0)
            
            # Use raw counts for naming (stored in cell data)
            if 'raw_counts' in cluster_cells.columns:
                raw_counts_matrix = np.vstack(cluster_cells['raw_counts'].values)
                avg_raw = np.mean(raw_counts_matrix, axis=0)
                total_raw = avg_raw.sum()
                poi_distribution = avg_raw / (total_raw + 1e-10)
            else:
                poi_distribution = avg_features[:7]
            
            # Find dominant NON-transportation category
            non_transport_dist = poi_distribution.copy()
            non_transport_dist[0] = 0  # Zero out transportation
            
            if non_transport_dist.sum() > 0.1:  # If there's meaningful non-transport activity
                dominant_poi_idx = np.argmax(non_transport_dist)
            else:
                dominant_poi_idx = np.argmax(poi_distribution)
            
            dominant_poi = self.poi_categories[dominant_poi_idx]
            dominant_ratio = poi_distribution[dominant_poi_idx]
            poi_density = avg_features[7]
            diversity = avg_features[8]
            
            cluster_name = self._name_cluster(
                dominant_poi, dominant_ratio, poi_density, 
                diversity, poi_distribution, avg_features
            )
            
            self.cluster_names[cluster_id] = cluster_name
            
            print(f"\nCluster {cluster_id}: {cluster_name}")
            print(f"  - Cells: {len(cluster_cells):,}")
            print(f"  - POI Distribution:")
            for i, cat in enumerate(self.poi_categories):
                print(f"      {cat}: {poi_distribution[i]:.1%}")
    
    def _name_cluster(self, dominant_poi, dominant_ratio, poi_density, diversity, poi_dist, features):
        """Improved auto-naming that handles transportation dominance"""
        transport_ratio = poi_dist[0]
        public_services_ratio = poi_dist[1]
        business_ratio = poi_dist[2]
        tourism_ratio = poi_dist[3]
        infrastructure_ratio = poi_dist[4]
        residential_ratio = poi_dist[5]
        
        density_level = np.expm1(poi_density)
        
        # Calculate non-transportation activity
        non_transport_activity = 1 - transport_ratio
        
        # Check for meaningful activity beyond just highways
        has_significant_activity = non_transport_activity > 0.3
        
        # Tourism areas (strong indicator)
        if tourism_ratio > 0.15:
            if business_ratio > 0.2:
                return "Tourism & Commercial"
            return "Tourism & Entertainment"
        
        # Business/Commercial areas
        if business_ratio > 0.25:
            if density_level > 50:
                return "Commercial Urban (High Density)"
            elif density_level > 20:
                return "Commercial Urban (Medium Density)"
            elif residential_ratio > 0.15:
                return "Mixed Commercial-Residential"
            else:
                return "Commercial Suburban"
        
        # Residential areas
        if residential_ratio > 0.25:
            if business_ratio > 0.15:
                return "Residential-Commercial Mixed"
            elif density_level > 30:
                return "Residential Urban"
            else:
                return "Residential Suburban"
        
        # Public services dominant
        if public_services_ratio > 0.2:
            if business_ratio > 0.15:
                return "Civic & Commercial"
            return "Public Services Area"
        
        # Only classify as Transportation Hub if truly dominant AND low other activity
        if transport_ratio > 0.5 and non_transport_activity < 0.3:
            if infrastructure_ratio > 0.15:
                return "Transportation & Industrial Corridor"
            return "Transportation Corridor"
        
        # Mixed use with good diversity
        if diversity > 1.2 and has_significant_activity:
            if density_level > 40:
                return "Mixed-Use Urban"
            else:
                return "Mixed-Use Suburban"
        
        # Infrastructure/Industrial
        if infrastructure_ratio > 0.2:
            return "Industrial / Infrastructure"
        
        # Low density areas
        if density_level < 10:
            if transport_ratio > 0.4:
                return "Rural Highway Corridor"
            return "Rural / Low Density"
        
        # Highway-adjacent areas with some activity
        if transport_ratio > 0.35 and has_significant_activity:
            if business_ratio > 0.1:
                return "Highway Commercial"
            elif residential_ratio > 0.1:
                return "Highway Residential"
            else:
                return "Highway Adjacent Mixed"
        
        # Default based on secondary characteristics
        if business_ratio > residential_ratio:
            return "Suburban Commercial"
        elif residential_ratio > 0.1:
            return "Suburban Residential"
        else:
            return f"Mixed Area ({dominant_poi})"
    
    def predict(self, lat, lon):
        """Fast prediction for coordinates"""
        geohash = pgh.encode(lat, lon, precision=self.precision)
        
        if geohash in self.cell_clusters:
            cluster_id = self.cell_clusters[geohash]
            return {
                'geohash': geohash,
                'cluster_id': cluster_id,
                'environment_type': self.cluster_names.get(cluster_id, f"Cluster {cluster_id}"),
                'method': 'direct_lookup'
            }
        else:
            return {
                'geohash': geohash,
                'cluster_id': -1,
                'environment_type': 'Unknown',
                'method': 'not_found'
            }
    
    def predict_batch(self, coords):
        """Batch prediction for multiple coordinates"""
        results = []
        for lat, lon in coords:
            results.append(self.predict(lat, lon))
        return results
    
    def visualize_clusters(self, cell_df, save_path='cluster_map.png'):
        """Visualize clusters"""
        print("\nCreating visualization...")
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Sample if too many points
        if len(cell_df) > 50000:
            plot_df = cell_df.sample(50000)
        else:
            plot_df = cell_df
        
        scatter = axes[0].scatter(
            plot_df['longitude'], 
            plot_df['latitude'],
            c=plot_df['cluster'],
            cmap='tab10',
            alpha=0.6,
            s=10
        )
        axes[0].set_xlabel('Longitude')
        axes[0].set_ylabel('Latitude')
        axes[0].set_title('Cluster Distribution')
        plt.colorbar(scatter, ax=axes[0])
        
        cluster_sizes = cell_df['cluster'].value_counts().sort_index()
        axes[1].bar(range(len(cluster_sizes)), cluster_sizes.values)
        axes[1].set_xlabel('Cluster ID')
        axes[1].set_ylabel('Number of Cells')
        axes[1].set_title('Cluster Sizes')
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved: {save_path}")
    
    def save_model(self, filepath):
        """Save model"""
        model_data = {
            'clustering_model': self.clustering_model,
            'scaler': self.scaler,
            'cell_clusters': self.cell_clusters,
            'cluster_names': self.cluster_names,
            'precision': self.precision,
            'n_clusters': self.n_clusters,
            'category_weights': self.category_weights
        }
        joblib.dump(model_data, filepath)
        print(f"Model saved: {filepath}")
    
    def load_model(self, filepath):
        """Load model"""
        model_data = joblib.load(filepath)
        self.clustering_model = model_data['clustering_model']
        self.scaler = model_data['scaler']
        self.cell_clusters = model_data['cell_clusters']
        self.cluster_names = model_data['cluster_names']
        self.precision = model_data['precision']
        self.n_clusters = model_data['n_clusters']
        if 'category_weights' in model_data:
            self.category_weights = model_data['category_weights']
        print(f"Model loaded: {filepath}")

## Main Pipeline

In [ ]:
def run_pipeline(poi_filepath, n_clusters=7):
    """
    Main pipeline for environment classification
    
    Args:
        poi_filepath: Path to POI CSV with columns: latitude, longitude, category
        n_clusters: Number of environment clusters
    """
    print("="*60)
    print("FAST ENVIRONMENT CLASSIFICATION PIPELINE")
    print("Optimized for 40M+ POIs")
    print("="*60)
    
    # 1. Load data
    print("\n1. Loading POI data...")
    poi_df = pd.read_csv(poi_filepath)
    print(f"   Loaded {len(poi_df):,} POIs")
    print(f"   Categories: {poi_df['category'].nunique()}")
    
    # 2. Initialize classifier
    print("\n2. Initializing classifier...")
    classifier = FastUnsupervisedEnvironmentClassifier(
        geohash_precision=6,
        n_clusters=n_clusters,
        n_jobs=-1,
        chunk_size=500000
    )
    
    # 3. Aggregate by geohash
    print("\n3. Aggregating POIs by geohash...")
    cell_df = classifier.aggregate_poi_by_geohash(poi_df)
    
    # Free memory
    del poi_df
    gc.collect()
    
    # 4. Find optimal clusters (optional)
    print("\n4. Finding optimal clusters...")
    optimal_k = classifier.find_optimal_clusters(cell_df, max_clusters=12)
    
    # 5. Cluster
    print("\n5. Clustering cells...")
    cell_df = classifier.cluster_cells(
        cell_df, 
        method='minibatch_kmeans',
        n_clusters=n_clusters
    )
    
    # 6. Visualize
    print("\n6. Creating visualizations...")
    classifier.visualize_clusters(cell_df)
    
    # 7. Save
    print("\n7. Saving model...")
    classifier.save_model('environment_classifier.pkl')
    
    print("\n" + "="*60)
    print("PIPELINE COMPLETED!")
    print("="*60)
    
    return classifier, cell_df

## Run Pipeline

In [ ]:
# Update this path to your POI data
POI_FILE = 'path/to/your/poi_data.csv'

# Run pipeline
classifier, cell_df = run_pipeline(POI_FILE, n_clusters=7)

## Test Predictions

In [ ]:
# Test coordinates
test_coords = [
    (3.1478, 101.6953),
    (3.1167, 101.6500),
    (3.1412, 101.6931),
]

for lat, lon in test_coords:
    result = classifier.predict(lat, lon)
    print(f"({lat:.4f}, {lon:.4f}) -> {result['environment_type']}")

## Performance Tips for 40M+ POIs

1. **Memory**: Process in chunks, delete intermediate DataFrames
2. **Speed**: Use `geohash2` library, MiniBatchKMeans, parallel processing
3. **Storage**: Save cell_df to parquet for faster reloading
4. **GPU**: Consider cuML for GPU-accelerated clustering

In [ ]:
# Save results to parquet for faster reloading
cell_df_save = cell_df.drop('features', axis=1)
cell_df_save.to_parquet('cell_clusters.parquet', index=False)
print("Saved cell clusters to parquet")